# Rodovias em Dados: análise exploratória

Notebook reprodutível, com dados oficiais da PRF, para o estudo 2023–2025. Execute as células na raiz do repositório. Os dados brutos permanecem em `data/raw/` como arquivos ZIP.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.etl import load_year, build_summary

raw = Path("data/raw")
frames, audits = [], []
for year in (2023, 2024, 2025):
    frame, audit = load_year(year, raw)
    frames.append(frame)
    audits.append(audit)
df = pd.concat(frames, ignore_index=True)
print(f"{len(df):,} ocorrências", {a["year"]:a["rows_read"] for a in audits})

## 1. Conferência dos dados

Antes de interpretar tendências, verificamos duplicatas, integridade das datas, ausências e coordenadas geográficas.

In [ ]:
pd.DataFrame(audits)[["year","rows_read","duplicates_full","duplicates_id","invalid_date","invalid_coordinates","missing_classification"]]

## 2. Série anual e mensal

Os valores abaixo são calculados diretamente dos CSVs oficiais, sem copiar números de páginas jornalísticas.

In [ ]:
summary = build_summary(df, audits)
annual = pd.DataFrame(summary["annual"])
annual

In [ ]:
sns.set_theme(style="whitegrid", palette=["#327E78", "#C4833C"])
fig, ax = plt.subplots(figsize=(8,4))
sns.barplot(data=annual, x="year", y="accidents", ax=ax, color="#327E78")
ax.set(title="Sinistros registrados pela PRF, 2023–2025", xlabel="Ano", ylabel="Ocorrências")
plt.show()

## 3. Distribuição territorial e gravidade

A proporção de ocorrências fatais dentro de um grupo não é uma taxa de risco por exposição ao tráfego.

In [ ]:
pd.DataFrame(summary["top_states_2025"]).head(10)

In [ ]:
pd.DataFrame(summary["severity_2025"]["tipo_pista"])

## 4. Publicação e reprodução

Rode `python scripts/run_pipeline.py` para regenerar JSON, figuras SVG, mapa Folium, SQLite e conteúdo do artigo. A auditoria com hashes SHA-256 fica em `data/processed/auditoria.json`.

## 4. Horários e dias da semana

Os horários são validados a partir do campo `horario`. Os dias da semana são derivados de `data_inversa`, não de categorias digitadas. Contagens não equivalem a probabilidade por deslocamento.


In [ ]:
hourly = pd.DataFrame(summary["hourly"])
weekday = pd.DataFrame(summary["weekday"])
display(hourly.sort_values("accidents", ascending=False).head())
display(weekday)
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(hourly["hour"], hourly["accidents"], color="#327E78")
ax.set(xlabel="Hora do registro", ylabel="Sinistros (2023–2025)", title="Frequência horária de sinistros")
plt.show()


## 5. Gravidade por tipo de sinistro

Aqui o denominador é o total de sinistros **do mesmo tipo**. Assim, um tipo menos frequente pode ter participação fatal maior. O resultado é uma associação descritiva, não efeito causal.


In [ ]:
fatal_type = pd.DataFrame(summary["fatal_by_type_2025"])
fatal_type[["name", "accidents", "fatal_accidents", "fatal_rate"]].head(10)


## 6. Dispersão mensal e valores extremos

A média, mediana, desvio-padrão e quartis dos 36 totais mensais ajudam a descrever concentração e dispersão. Um mês extremo não deve ser interpretado como evidência de causa ou anomalia de registro sem investigação adicional.


In [ ]:
eda = summary["eda_summary"]
print("Média mensal:", eda["monthly_accidents_mean"])
print("Mediana mensal:", eda["monthly_accidents_median"])
print("Desvio-padrão populacional:", eda["monthly_accidents_std_population"])
print("Maior mês:", eda["monthly_highest"])
print("Menor mês:", eda["monthly_lowest"])


## 7. Integridade dos agregados

Conferimos se a distribuição territorial soma os mesmos totais das séries anuais e se não há diferença entre mortes e ocorrências com pelo menos uma morte.


In [ ]:
assert sum(x["accidents"] for x in summary["top_states_2025"]) == summary["annual"][-1]["accidents"]
assert sum(x["accidents"] for x in summary["weekday"]) == len(df)
assert sum(x["fatal_accidents"] for x in summary["annual"]) <= sum(x["deaths"] for x in summary["annual"])
print("Conferências aprovadas")
